# Chapter 26 — Evaluating and Testing AI Systems for Networks
## How Do You Know Your AI Is Actually Correct?

In traditional software: if the test suite passes, you have confidence the code works.
Tests are deterministic: given input X, the function returns exactly Y.

**AI systems don't work this way.**

An LLM that correctly answers 95% of your test questions is wrong 5% of the time —
and you don't know *which* questions those are until a user hits them.

For AI in network operations — where wrong answers can cause **configuration mistakes,
missed security incidents, or incorrect diagnostic conclusions** — this is not an
academic concern. It is **operational risk.**

| Metric | What It Measures | The Problem |
|--------|-----------------|-------------|
| **Exact match** | Character-by-character match | "BGP uses port 179" ≠ "TCP port 179 for BGP" → both correct, one scores 0 |
| **BLEU / ROUGE** | N-gram overlap with reference | Correct answer with different wording = low score |
| **Perplexity** | Model confidence | Fluent hallucination = low perplexity = "good" |
| **LLM-as-Judge** ✅ | Semantic correctness | Understands meaning, not just tokens |
| **RAGAS** ✅ | RAG-specific: faithfulness, recall | Diagnoses *where* the pipeline fails |

> **No external eval libraries needed.** All built on the Anthropic API.

### Install
```
pip install anthropic
```


## Setup — API Client and Test Suite

In [ ]:
import os, json, re, math
from anthropic import Anthropic
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()   # reads ANTHROPIC_API_KEY from env

HAIKU  = "claude-haiku-4-5-20251001"   # the model being evaluated
SONNET = "claude-sonnet-4-20250514"           # the judge model (smarter)

def ask(prompt, model=HAIKU, system="You are a network engineer."):
    r = client.messages.create(
        model=model, max_tokens=512, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Networking Q&A test suite ─────────────────────────────────────────────────
# Each entry: question, reference answer, retrieved context (for RAG tests)
TEST_SUITE = [
    {
        "id": "Q01",
        "question": "What TCP port does BGP use?",
        "reference": "BGP uses TCP port 179.",
        "context": "Border Gateway Protocol (BGP) is defined in RFC 4271. "
                   "BGP uses TCP as its transport protocol and listens on port 179.",
        "category": "protocol_fundamentals",
    },
    {
        "id": "Q02",
        "question": "What is the difference between OSPF area 0 and a stub area?",
        "reference": "Area 0 is the backbone area — all OSPF areas must connect to it. "
                     "A stub area blocks external LSAs (Type 5) and uses a default route "
                     "instead, reducing the LSDB size on stub routers.",
        "context": "OSPF uses a hierarchical area structure. Area 0 is the backbone. "
                   "Stub areas do not receive Type 5 External LSAs. Instead, the ABR "
                   "injects a default route into the stub area. This reduces the size "
                   "of the link-state database on routers inside the stub area.",
        "category": "ospf",
    },
    {
        "id": "Q03",
        "question": "How does OSPF MD5 authentication work?",
        "reference": "OSPF MD5 authentication generates an MD5 hash of each OSPF packet "
                     "combined with a shared key. The receiving router verifies the hash. "
                     "Both routers must have the same key ID and key string configured.",
        "context": "OSPF supports MD5 cryptographic authentication. A key-id and key "
                   "string must match on both sides of an adjacency. The MD5 hash is "
                   "computed over the OSPF packet contents plus the key.",
        "category": "security",
    },
    {
        "id": "Q04",
        "question": "What causes OSPF neighbors to be stuck in EXSTART?",
        "reference": "OSPF EXSTART is usually caused by an MTU mismatch between neighbors. "
                     "The routers cannot agree on the DBD packet size. Other causes: "
                     "authentication mismatch, duplicate router IDs.",
        "context": "The EXSTART state is where OSPF neighbors negotiate the master/slave "
                   "relationship and starting sequence number for DBD packets. A common "
                   "cause of being stuck in EXSTART is MTU mismatch — if one side sends "
                   "a DBD larger than the other's MTU, the packet is dropped.",
        "category": "troubleshooting",
    },
    {
        "id": "Q05",
        "question": "What is BGP route reflector and why is it used?",
        "reference": "A BGP route reflector allows IBGP routes to be reflected to other "
                     "IBGP peers, eliminating the need for a full mesh of IBGP sessions. "
                     "Without route reflectors, every IBGP router must peer with every "
                     "other IBGP router (O(n²) sessions).",
        "context": "In IBGP, a router does not re-advertise routes learned from one IBGP "
                   "peer to another IBGP peer (split-horizon rule). A route reflector "
                   "breaks this rule for its clients, reflecting routes between them. "
                   "This allows large networks to avoid full IBGP mesh (n*(n-1)/2 sessions).",
        "category": "bgp",
    },
]

print(f"Test suite loaded: {len(TEST_SUITE)} questions")
for q in TEST_SUITE:
    print(f"  [{q['id']}] [{q['category']}] {q['question'][:60]}...")


---
## Demo 1 — Why Traditional Metrics Fail: The Semantic Gap

Watch exact match and simple word overlap fail on answers that are **semantically correct**:

```
Question: "What TCP port does BGP use?"

Reference:  "BGP uses TCP port 179."
AI answer:  "BGP operates on port 179 over TCP."  ← semantically identical

Exact match: 0.0   (character mismatch)
Word overlap: 0.4  (shared words: BGP, port, 179, TCP — but different structure)
LLM judge:   9/10  (understands they mean the same thing)
```

The problem: exact match and word overlap measure **surface properties** of text.
They cannot recognize that two differently-worded answers are both correct.

This is exactly why BLEU score is useless for evaluating network AI answers.


In [ ]:
# ── Traditional Metrics vs LLM Judge ─────────────────────────────────────────

def exact_match(generated: str, reference: str) -> float:
    """Returns 1.0 if strings match exactly (case-insensitive), 0.0 otherwise."""
    return 1.0 if generated.strip().lower() == reference.strip().lower() else 0.0

def word_overlap_f1(generated: str, reference: str) -> float:
    """
    Token-level F1 (similar to ROUGE-1 recall+precision).
    Counts shared words between generated and reference.
    """
    gen_tokens = set(generated.lower().split())
    ref_tokens = set(reference.lower().split())
    common = gen_tokens & ref_tokens
    if not common:
        return 0.0
    precision = len(common) / len(gen_tokens)
    recall    = len(common) / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def llm_judge_score(question: str, generated: str, reference: str) -> dict:
    """
    LLM-as-a-Judge: Sonnet evaluates Haiku's answer on correctness and completeness.
    Returns scores 1-10 with reasoning.
    """
    prompt = f"""Evaluate this AI-generated answer for a network engineering question.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Score the generated answer on:
1. Factual accuracy (1-10): is the technical information correct?
2. Completeness (1-10): does it cover the key points in the reference?
3. Clarity (1-10): is it clear and unambiguous?

Return ONLY JSON:
{{
  "factual_accuracy": 8,
  "completeness": 7,
  "clarity": 9,
  "overall": 8,
  "reasoning": "one sentence explaining the score"
}}
"""
    raw = ask(prompt, model=SONNET,
              system="You are a strict technical evaluator for network engineering AI systems. Return only JSON.")
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return {"factual_accuracy": 0, "completeness": 0, "clarity": 0, "overall": 0,
            "reasoning": "parse error"}


# Generate answers with Haiku for all test questions
print("Generating answers with Haiku (model under test)...")
print("=" * 60)

results = []
for q in TEST_SUITE:
    generated = ask(q["question"], model=HAIKU,
                    system="You are a network engineering expert. Answer concisely.")
    em_score  = exact_match(generated, q["reference"])
    wf1_score = word_overlap_f1(generated, q["reference"])
    judge     = llm_judge_score(q["question"], generated, q["reference"])

    results.append({
        "id": q["id"],
        "question": q["question"],
        "generated": generated,
        "reference": q["reference"],
        "exact_match": em_score,
        "word_f1": round(wf1_score, 2),
        "llm_judge": judge,
    })

    print(f"\n[{q['id']}] {q['question'][:55]}...")
    print(f"  Generated : {generated[:80]}...")
    print(f"  Reference : {q['reference'][:80]}...")
    print(f"  Exact match  : {em_score:.1f}   ← usually 0 even for correct answers")
    print(f"  Word F1 (ROUGE-1): {wf1_score:.2f}")
    print(f"  LLM Judge overall: {judge.get('overall', '?')}/10  ← {judge.get('reasoning','')}")

# Summary
avg_em    = sum(r["exact_match"] for r in results) / len(results)
avg_wf1   = sum(r["word_f1"] for r in results) / len(results)
avg_judge = sum(r["llm_judge"].get("overall", 0) for r in results) / len(results)
print(f"\n{'='*60}")
print(f"EVALUATION SUMMARY ({len(results)} questions)")
print(f"  Average Exact Match  : {avg_em:.2f}  ← misleadingly low")
print(f"  Average Word F1      : {avg_wf1:.2f}  ← still surface-level")
print(f"  Average LLM Judge    : {avg_judge:.1f}/10  ← semantically correct")
print(f"\nConclusion: Exact match underestimates quality by {(avg_judge/10 - avg_em)*100:.0f}%")


---
## Demo 2 — RAGAS-Style Evaluation: Faithfulness and Context Recall

**RAGAS** evaluates the complete RAG pipeline, not just the final answer.

### Faithfulness — Is every claim in the answer supported by the retrieved docs?
```
Answer claim: "BGP uses TCP port 179"
Context:      "BGP listens on port 179"     → ENTAILMENT → claim is grounded ✓

Answer claim: "BGP also supports UDP"
Context:      (nothing about UDP)           → NEUTRAL → hallucination risk ✗
```
`Faithfulness = grounded_claims / total_claims`

### Context Recall — Did retrieval find the right documents?
```
Reference answer has 3 key facts:
  Fact 1: "Area 0 is backbone"        → found in context ✓
  Fact 2: "Stub areas block Type 5"   → found in context ✓
  Fact 3: "Default route is injected" → NOT found in context ✗

Context Recall = 2/3 = 0.67  → retrieval missed one key fact
```
Low context recall means **fix the retriever, not the LLM**.


In [ ]:
# ── RAGAS-Style Evaluation (Faithfulness + Context Recall) ───────────────────

def decompose_into_claims(text: str) -> list[str]:
    """Ask LLM to decompose text into atomic factual claims."""
    prompt = f"""Break this text into individual, atomic factual claims.
Each claim should be a single, standalone statement of fact.

Text: {text}

Return ONLY a JSON array of strings:
["claim 1", "claim 2", "claim 3"]
"""
    raw = ask(prompt, model=HAIKU,
              system="You are a fact decomposition system. Return only JSON.")
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    # Fallback: split by sentences
    return [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

def check_claim_groundedness(claim: str, context: str) -> str:
    """
    NLI-style check: is this claim supported by the context?
    Returns: ENTAILMENT / CONTRADICTION / NEUTRAL
    """
    prompt = f"""Given this context, classify whether the claim is supported.

Context: {context}

Claim: {claim}

Choose ONE:
- ENTAILMENT: context clearly supports this claim
- CONTRADICTION: context contradicts this claim
- NEUTRAL: context doesn't mention this claim

Return ONLY one word: ENTAILMENT, CONTRADICTION, or NEUTRAL
"""
    result = ask(prompt, model=HAIKU,
                 system="You are a fact-checking system. Return only one word.")
    result = result.strip().upper()
    if "ENTAILMENT" in result:   return "ENTAILMENT"
    if "CONTRADICTION" in result: return "CONTRADICTION"
    return "NEUTRAL"

def compute_faithfulness(generated: str, context: str) -> dict:
    """
    Faithfulness = grounded claims / total claims.
    Scores how much of the answer is supported by the retrieved context.
    """
    claims = decompose_into_claims(generated)
    if not claims:
        return {"score": 0.0, "claims": [], "grounded": 0, "total": 0}

    grounded = 0
    claim_results = []
    for claim in claims:
        verdict = check_claim_groundedness(claim, context)
        if verdict == "ENTAILMENT":
            grounded += 1
        claim_results.append({"claim": claim, "verdict": verdict})

    return {
        "score": round(grounded / len(claims), 2),
        "claims": claim_results,
        "grounded": grounded,
        "total": len(claims),
    }

def compute_context_recall(reference: str, context: str) -> dict:
    """
    Context Recall = reference facts found in context / total reference facts.
    Diagnoses retrieval quality — did we retrieve the right documents?
    """
    ref_claims = decompose_into_claims(reference)
    if not ref_claims:
        return {"score": 0.0, "supported": 0, "total": 0}

    supported = 0
    for claim in ref_claims:
        verdict = check_claim_groundedness(claim, context)
        if verdict == "ENTAILMENT":
            supported += 1

    return {
        "score": round(supported / len(ref_claims), 2),
        "supported": supported,
        "total": len(ref_claims),
    }


# Evaluate two questions from our test suite
print("RAGAS-STYLE EVALUATION")
print("=" * 60)

for q in TEST_SUITE[:2]:
    generated = ask(q["question"], model=HAIKU,
                    system="You are a network engineer. Answer based on the context.",)

    print(f"\n[{q['id']}] {q['question']}")
    print(f"  Context snippet: {q['context'][:100]}...")
    print(f"  Generated: {generated[:100]}...")

    # Faithfulness
    faith = compute_faithfulness(generated, q["context"])
    print(f"\n  FAITHFULNESS: {faith['score']:.0%} ({faith['grounded']}/{faith['total']} claims grounded)")
    for c in faith["claims"]:
        icon = "✓" if c["verdict"] == "ENTAILMENT" else ("✗" if c["verdict"] == "CONTRADICTION" else "?")
        print(f"    {icon} [{c['verdict']}] {c['claim'][:70]}")

    # Context Recall
    recall = compute_context_recall(q["reference"], q["context"])
    print(f"\n  CONTEXT RECALL: {recall['score']:.0%} ({recall['supported']}/{recall['total']} reference facts in context)")
    if recall['score'] < 0.8:
        print(f"  ⚠ Low recall → fix the retriever, not the LLM")
    else:
        print(f"  ✓ Good recall → context contains enough information")


---
## Demo 3 — Hallucination Detection: Two Methods

### Method 1 — Self-Consistency
Ask the same question 3 times at higher temperature (adds variation).
If the answers **contradict each other**, the model is likely hallucinating.
If they agree, it's probably correct.

```
Run 1: "BGP uses TCP port 179"
Run 2: "BGP uses TCP port 179"
Run 3: "BGP uses UDP port 179"   ← CONTRADICTION → hallucination risk
```

### Method 2 — Groundedness Check
For RAG answers: verify each claim in the answer against the retrieved documents.
Any claim that is **NEUTRAL or CONTRADICTION** with the context is a potential hallucination.

This is the practical implementation of what RAGAS calls "faithfulness" — just focused
specifically on catching invented facts.


In [ ]:
# ── Hallucination Detection ───────────────────────────────────────────────────

def self_consistency_check(question: str, n_runs: int = 3) -> dict:
    """
    Run the same question n times, check if answers agree.
    Disagreement = hallucination risk.
    """
    answers = []
    for i in range(n_runs):
        # Slightly varied prompt to encourage different phrasings
        ans = ask(f"{question}\n(Run {i+1} — answer directly and concisely)",
                  model=HAIKU,
                  system="You are a network engineer. Answer factually and concisely.")
        answers.append(ans)

    # Ask LLM to compare the runs
    compare_prompt = f"""Compare these {n_runs} answers to the same question.

Question: {question}

Answers:
{chr(10).join(f'Run {i+1}: {a}' for i, a in enumerate(answers))}

Analyze:
1. Do all answers agree on the key facts?
2. Are there any contradictions?
3. Hallucination risk: LOW / MEDIUM / HIGH

Return JSON:
{{
  "answers_agree": true,
  "contradictions": ["describe any contradiction found"],
  "hallucination_risk": "LOW",
  "consistent_facts": ["facts all runs agree on"],
  "reasoning": "brief explanation"
}}
"""
    raw = ask(compare_prompt, model=SONNET,
              system="You are a hallucination detection system. Return only JSON.")
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    result = {}
    if match:
        try:
            result = json.loads(match.group())
        except Exception:
            result = {"hallucination_risk": "UNKNOWN"}

    return {"answers": answers, "analysis": result}


def groundedness_check(generated: str, context: str) -> dict:
    """
    Groundedness: what fraction of the answer's claims are supported by context?
    Returns flagged claims that might be hallucinated.
    """
    faith = compute_faithfulness(generated, context)
    ungrounded = [c for c in faith["claims"]
                  if c["verdict"] in ("NEUTRAL", "CONTRADICTION")]
    return {
        "faithfulness_score": faith["score"],
        "ungrounded_claims": ungrounded,
        "is_grounded": faith["score"] >= 0.8,
    }


# Test on questions where hallucination is more likely
print("HALLUCINATION DETECTION")
print("=" * 60)

# Self-consistency check
tricky_question = "What is the exact OSPF hello interval for a broadcast network in Cisco IOS?"
print(f"\n[Method 1] Self-Consistency Check")
print(f"Question: {tricky_question}")
sc_result = self_consistency_check(tricky_question, n_runs=3)

print(f"\nRun answers:")
for i, ans in enumerate(sc_result["answers"], 1):
    print(f"  Run {i}: {ans}")

analysis = sc_result["analysis"]
risk = analysis.get("hallucination_risk", "UNKNOWN")
icon = {"LOW": "✅", "MEDIUM": "⚠", "HIGH": "🔴"}.get(risk, "?")
print(f"\nHallucination risk: {icon} {risk}")
print(f"Answers agree: {analysis.get('answers_agree', '?')}")
if analysis.get('contradictions'):
    print(f"Contradictions: {analysis['contradictions']}")
print(f"Reasoning: {analysis.get('reasoning', '')}")

# Groundedness check
print(f"\n[Method 2] Groundedness Check (RAG answer vs context)")
test_q = TEST_SUITE[3]   # OSPF EXSTART question
generated_answer = ask(test_q["question"], model=HAIKU)
ground_result = groundedness_check(generated_answer, test_q["context"])

print(f"Question: {test_q['question']}")
print(f"Generated: {generated_answer[:100]}...")
print(f"\nGroundedness score: {ground_result['faithfulness_score']:.0%}")
if ground_result["ungrounded_claims"]:
    print(f"⚠ Potentially hallucinated claims ({len(ground_result['ungrounded_claims'])}):")
    for c in ground_result["ungrounded_claims"]:
        print(f"  [{c['verdict']}] {c['claim'][:80]}")
else:
    print("✅ All claims grounded in context — no hallucination detected")
print(f"Overall: {'✅ Grounded' if ground_result['is_grounded'] else '⚠ Review required'}")


---
## Demo 4 — Answer Correctness: F1 Over Atomic Statements

**Answer Correctness** is an F1 score computed over atomic factual statements.
This solves the problem of BLEU/ROUGE: instead of word overlap, we compare *facts*.

```
Reference: "BGP uses TCP. Port is 179. Both routers must agree."
Generated: "BGP uses TCP port 179. The session uses a 3-way handshake."

TP = {TCP, port 179}              ← in both
FP = {3-way handshake}            ← in generated, not in reference
FN = {both routers must agree}    ← in reference, not in generated

Precision = TP/(TP+FP) = 2/3 = 0.67
Recall    = TP/(TP+FN) = 2/3 = 0.67
F1        = 0.67
```

Unlike BLEU, this correctly handles:
- Paraphrasing (semantics checked by LLM, not tokens)
- Partial correctness (partial credit through F1)
- Completeness (recall penalizes missing facts)


In [ ]:
# ── Answer Correctness (F1 over Atomic Statements) ───────────────────────────

def classify_claim_overlap(gen_claims: list[str],
                            ref_claims: list[str]) -> dict:
    """
    Compare generated and reference claims:
    TP: gen claim matches a ref claim
    FP: gen claim not in reference (extra / wrong)
    FN: ref claim not in generated (missing)
    """
    prompt = f"""Compare two sets of claims about a networking topic.

Generated claims:
{json.dumps(gen_claims, indent=2)}

Reference claims:
{json.dumps(ref_claims, indent=2)}

Classify each generated claim as:
- TP (True Positive): semantically matches a reference claim
- FP (False Positive): not in reference (extra or incorrect)

And list any reference claims NOT covered by the generated claims:
- FN (False Negative): in reference but missing from generated

Return ONLY JSON:
{{
  "tp_claims": ["..."],
  "fp_claims": ["..."],
  "fn_claims": ["..."]
}}
"""
    raw = ask(prompt, model=SONNET,
              system="You are a fact comparison system. Return only valid JSON.")
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return {"tp_claims": [], "fp_claims": gen_claims, "fn_claims": ref_claims}

def answer_correctness_f1(generated: str, reference: str) -> dict:
    """Full answer correctness score using F1 over atomic claims."""
    gen_claims = decompose_into_claims(generated)
    ref_claims = decompose_into_claims(reference)

    overlap    = classify_claim_overlap(gen_claims, ref_claims)
    tp = len(overlap.get("tp_claims", []))
    fp = len(overlap.get("fp_claims", []))
    fn = len(overlap.get("fn_claims", []))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "f1": round(f1, 2),
        "precision": round(precision, 2),
        "recall": round(recall, 2),
        "tp": tp, "fp": fp, "fn": fn,
        "tp_claims": overlap.get("tp_claims", []),
        "fp_claims": overlap.get("fp_claims", []),
        "fn_claims": overlap.get("fn_claims", []),
    }


print("ANSWER CORRECTNESS — F1 over Atomic Statements")
print("=" * 60)

for q in TEST_SUITE[1:3]:   # Test on OSPF and MD5 questions
    generated = ask(q["question"], model=HAIKU,
                    system="You are a network engineer. Answer thoroughly.")

    scores = answer_correctness_f1(generated, q["reference"])

    print(f"\n[{q['id']}] {q['question']}")
    print(f"  Generated: {generated[:100]}...")

    print(f"\n  Answer Correctness F1: {scores['f1']:.2f}")
    print(f"  Precision: {scores['precision']:.2f}  Recall: {scores['recall']:.2f}")
    print(f"  TP={scores['tp']}  FP={scores['fp']}  FN={scores['fn']}")

    if scores["tp_claims"]:
        print(f"  ✓ Correct facts ({scores['tp']}):")
        for c in scores["tp_claims"][:2]:
            print(f"    + {c}")
    if scores["fp_claims"]:
        print(f"  ✗ Extra/incorrect ({scores['fp']}):")
        for c in scores["fp_claims"][:2]:
            print(f"    - {c}")
    if scores["fn_claims"]:
        print(f"  ⚠ Missing from answer ({scores['fn']}):")
        for c in scores["fn_claims"][:2]:
            print(f"    ? {c}")


---
## Demo 5 — Full Evaluation Pipeline: Quality Report

A complete evaluation run that generates a **quality scorecard** for your AI system.

```
Test Suite (Q&A pairs)
        │
   For each question:
   ├── Generate answer (model under test)
   ├── LLM Judge score (1-10: accuracy, completeness, clarity)
   ├── Faithfulness (claims grounded in context?)
   ├── Context Recall (retrieval found right docs?)
   ├── Answer Correctness F1 (fact-level overlap)
   └── Hallucination risk (self-consistency)
        │
   Quality Report Card
   ├── Per-dimension scores
   ├── Weakest questions identified
   └── Specific improvement recommendations
```

This is **evaluation-driven development**: measure first, then improve.


In [ ]:
# ── Full Evaluation Pipeline ──────────────────────────────────────────────────

def run_full_evaluation(test_suite: list[dict],
                        model_under_test: str = HAIKU) -> dict:
    """
    Run comprehensive evaluation across all test questions.
    Returns a structured quality report.
    """
    print(f"Running full evaluation for model: {model_under_test}")
    print(f"Test suite: {len(test_suite)} questions")
    print("=" * 70)

    all_results = []

    for q in test_suite:
        print(f"\n  Evaluating [{q['id']}] {q['question'][:50]}...")

        # Generate answer
        generated = ask(q["question"], model=model_under_test,
                        system="You are a network engineering expert. Answer clearly and accurately.")

        # LLM Judge
        judge = llm_judge_score(q["question"], generated, q["reference"])

        # Faithfulness (with context)
        faith = compute_faithfulness(generated, q["context"])

        # Context Recall
        recall = compute_context_recall(q["reference"], q["context"])

        # Answer Correctness
        correctness = answer_correctness_f1(generated, q["reference"])

        result = {
            "id":          q["id"],
            "category":    q["category"],
            "question":    q["question"],
            "generated":   generated,
            "judge_score": judge.get("overall", 0),
            "faithfulness": faith["score"],
            "context_recall": recall["score"],
            "answer_f1":   correctness["f1"],
            "fp_count":    correctness["fp"],
            "fn_count":    correctness["fn"],
        }
        all_results.append(result)

        print(f"    Judge: {result['judge_score']}/10 | "
              f"Faith: {result['faithfulness']:.0%} | "
              f"Recall: {result['context_recall']:.0%} | "
              f"F1: {result['answer_f1']:.2f}")

    # ── Aggregate scores ──────────────────────────────────────────────────────
    def avg(key): return sum(r[key] for r in all_results) / len(all_results)

    report = {
        "model":           model_under_test,
        "n_questions":     len(test_suite),
        "avg_judge_score": round(avg("judge_score"), 1),
        "avg_faithfulness": round(avg("faithfulness"), 2),
        "avg_context_recall": round(avg("context_recall"), 2),
        "avg_answer_f1":   round(avg("answer_f1"), 2),
        "results":         all_results,
    }

    # ── Quality Report Card ───────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("AI SYSTEM QUALITY REPORT CARD")
    print("=" * 70)
    print(f"Model:           {model_under_test}")
    print(f"Questions tested: {report['n_questions']}")
    print()

    metrics = [
        ("LLM Judge (accuracy)",  report["avg_judge_score"],   10.0, "/10"),
        ("Faithfulness (RAG)",     report["avg_faithfulness"],   1.0, ""),
        ("Context Recall",         report["avg_context_recall"], 1.0, ""),
        ("Answer Correctness F1",  report["avg_answer_f1"],      1.0, ""),
    ]

    for name, score, max_val, suffix in metrics:
        pct    = score / max_val
        bar    = "█" * int(pct * 20)
        grade  = "A" if pct >= 0.9 else "B" if pct >= 0.8 else "C" if pct >= 0.7 else "D"
        unit   = f"{score:.0%}" if max_val == 1.0 else f"{score:.1f}{suffix}"
        print(f"  {name:<28} [{bar:<20}] {unit}  Grade: {grade}")

    # Weakest questions
    sorted_by_f1 = sorted(all_results, key=lambda r: r["answer_f1"])
    print(f"\nWeakest questions (lowest Answer F1):")
    for r in sorted_by_f1[:2]:
        print(f"  [{r['id']}] F1={r['answer_f1']:.2f} | {r['question'][:60]}...")
        if r["fn_count"] > 0:
            print(f"    → Missing {r['fn_count']} key fact(s) — consider adding to context")
        if r["fp_count"] > 0:
            print(f"    → Generated {r['fp_count']} extra claim(s) — possible hallucination")

    # LLM-generated improvement recommendations
    print("\n[LLM Recommendations]")
    rec_prompt = f"""Based on this AI evaluation report for a network engineering assistant:

Scores:
  LLM Judge: {report['avg_judge_score']}/10
  Faithfulness: {report['avg_faithfulness']:.0%}
  Context Recall: {report['avg_context_recall']:.0%}
  Answer F1: {report['avg_answer_f1']:.0%}

Weakest area: {min(metrics, key=lambda m: m[1]/m[2])[0]}

Write 3 specific, actionable improvement recommendations.
Each in one sentence. Focus on practical fixes.
"""
    recommendations = ask(rec_prompt, model=HAIKU,
                           system="You are an AI systems evaluator. Give specific, practical recommendations.")
    print(recommendations)

    return report


# Run the full evaluation
quality_report = run_full_evaluation(TEST_SUITE)


---
## Summary — The AI Evaluation Stack for Network Systems

### Why Traditional Metrics Fail

```
"BGP uses TCP port 179"  vs  "TCP port 179 for BGP"
  Exact match: 0.0   ← WRONG — both are correct
  Word F1:     0.57  ← misleadingly low
  LLM Judge:   9/10  ← correctly captures semantic equivalence
```

### The RAGAS Evaluation Framework

| Metric | What It Measures | When Low → Fix |
|--------|-----------------|----------------|
| **Faithfulness** | Claims grounded in context | LLM is hallucinating |
| **Context Recall** | Reference facts in retrieved docs | Fix the retriever |
| **Answer Correctness (F1)** | Fact-level overlap with reference | Improve prompting or context |

### Hallucination Detection Methods

| Method | How | When to Use |
|--------|-----|-------------|
| **Self-Consistency** | Run 3× at higher temp, check agreement | Factual questions |
| **Groundedness** | NLI: claim vs context | RAG systems |
| **LLM-as-Judge** | Stronger model reviews weaker model | Open-ended answers |

### Evaluation-Driven Development

```
Define test suite FIRST
    ↓
Baseline evaluation (what score does the current system get?)
    ↓
Identify weakest dimension (faithfulness? recall? correctness?)
    ↓
Fix the right component (retriever, chunking, prompt, model)
    ↓
Re-evaluate → measure improvement
    ↓
Deploy only when scores meet threshold
```

### What's Next

- **Chapter 27**: Production Deployment — moving your evaluated, tested AI network
  system from notebook to production infrastructure with monitoring, scaling, and
  safety guardrails

> You cannot improve what you don't measure.
> And you cannot measure AI quality with exact match.
> The evaluation stack in this chapter is what separates AI prototypes from
> production-grade network automation systems.
